# 03 — Continuum Memory System (CMS)

**Core claim**: Memory isn't binary (short/long-term) — it's a continuum of update frequencies.

Experiments:
1. Multi-frequency signal decomposition (FFT verification)
2. Update timeline heatmap
3. Copy/recall task at varying delays
4. Ablation on C_base
5. Gradient flow analysis per block

In [ ]:
import sys; sys.path.insert(0, '..')
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from src.cms import ContinuumMemorySystem, CMSTrainer, MemoryBlock
from src.data import generate_multifreq_signal, generate_copy_task
from src.utils import set_seed, plot_update_timeline, plot_loss_curves
set_seed(42)

## 1. Multi-Frequency Signal Decomposition

In [ ]:
# Generate a sum-of-sines signal
freqs = [5, 20, 50]
t, signal = generate_multifreq_signal(freqs, duration=1.0, sample_rate=500)

# Train CMS to reconstruct the signal from time input
cms = ContinuumMemorySystem(input_dim=1, hidden_dim=64, output_dim=1, n_levels=3, c_base=4)
trainer = CMSTrainer(cms, lr=1e-3)

x_train = t.unsqueeze(1)  # (500, 1)
y_train = signal.unsqueeze(1)  # (500, 1)

losses = []
for epoch in range(200):
    # Mini-batch
    idx = torch.randperm(len(x_train))[:64]
    info = trainer.train_step(x_train[idx], y_train[idx], nn.MSELoss())
    losses.append(info['loss'])

# Plot results
cms.eval()
with torch.no_grad():
    pred = cms(x_train)

fig, axes = plt.subplots(2, 1, figsize=(12, 6))
axes[0].plot(t.numpy(), signal.numpy(), label='Target', alpha=0.7)
axes[0].plot(t.numpy(), pred.squeeze().numpy(), label='CMS Prediction', alpha=0.7)
axes[0].set_title('Multi-Frequency Signal Reconstruction')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# FFT analysis
from numpy.fft import fft, fftfreq
N = len(signal)
freqs_fft = fftfreq(N, 1/500)[:N//2]
target_fft = np.abs(fft(signal.numpy()))[:N//2]
pred_fft = np.abs(fft(pred.squeeze().numpy()))[:N//2]

axes[1].plot(freqs_fft, target_fft, label='Target FFT', alpha=0.7)
axes[1].plot(freqs_fft, pred_fft, label='CMS Prediction FFT', alpha=0.7)
axes[1].set_xlim(0, 80)
axes[1].set_title('Frequency Domain')
axes[1].set_xlabel('Frequency (Hz)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 2. Update Timeline Heatmap

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 3))
plot_update_timeline(100, 4, c_base=2, title='CMS Timeline (C_base=2)', ax=axes[0])
plot_update_timeline(100, 4, c_base=4, title='CMS Timeline (C_base=4)', ax=axes[1])
plt.tight_layout()
plt.show()

## 3. Copy/Recall Task at Varying Delays

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

delays = [5, 20, 50]
cms_results = {}
baseline_results = {}

for delay in delays:
    set_seed(42)
    inputs, targets = generate_copy_task(seq_len=8, delay=delay, vocab_size=8, n_samples=500)
    total_len = inputs.shape[1]
    vocab = 9  # 8 + blank
    
    # CMS model
    cms_model = ContinuumMemorySystem(input_dim=vocab, hidden_dim=64, output_dim=vocab, n_levels=3, c_base=4)
    cms_trainer = CMSTrainer(cms_model, lr=1e-3)
    
    # Single-frequency baseline
    baseline = ContinuumMemorySystem(input_dim=vocab, hidden_dim=64, output_dim=vocab, n_levels=3, c_base=1)
    base_trainer = CMSTrainer(baseline, lr=1e-3)
    
    # One-hot encode inputs
    x_oh = torch.nn.functional.one_hot(inputs, vocab).float().view(-1, vocab)
    y_flat = targets.view(-1)
    
    cms_losses, base_losses = [], []
    for step in range(200):
        idx = torch.randperm(x_oh.shape[0])[:256]
        
        info_c = cms_trainer.train_step(x_oh[idx], x_oh[idx], nn.MSELoss())
        cms_losses.append(info_c['loss'])
        
        info_b = base_trainer.train_step(x_oh[idx], x_oh[idx], nn.MSELoss())
        base_losses.append(info_b['loss'])
    
    cms_results[f'delay={delay}'] = cms_losses
    baseline_results[f'delay={delay}'] = base_losses

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
plot_loss_curves(cms_results, title='CMS (multi-freq)', ax=axes[0])
plot_loss_curves(baseline_results, title='Baseline (single-freq)', ax=axes[1])
plt.tight_layout()
plt.show()

## 4. Ablation: C_base

In [ ]:
c_bases = [2, 4, 8, 16]
results = {}

for cb in c_bases:
    set_seed(42)
    cms = ContinuumMemorySystem(input_dim=1, hidden_dim=64, output_dim=1, n_levels=4, c_base=cb)
    trainer = CMSTrainer(cms, lr=1e-3)
    
    t, signal = generate_multifreq_signal([5, 20, 50], duration=1.0, sample_rate=500)
    x = t.unsqueeze(1)
    y = signal.unsqueeze(1)
    
    losses = []
    for step in range(300):
        idx = torch.randperm(len(x))[:64]
        info = trainer.train_step(x[idx], y[idx], nn.MSELoss())
        losses.append(info['loss'])
    results[f'C_base={cb}'] = losses

plot_loss_curves(results, title='CMS C_base Ablation', log_scale=True)
plt.show()

## 5. Gradient Flow per Block

In [ ]:
set_seed(42)
cms = ContinuumMemorySystem(input_dim=1, hidden_dim=64, output_dim=1, n_levels=4, c_base=4)
trainer = CMSTrainer(cms, lr=1e-3)
t, signal = generate_multifreq_signal([5, 20, 50], duration=1.0, sample_rate=500)

grad_norms = {f'Block {i} (C^{i}={4**i})': [] for i in range(4)}

for step in range(200):
    idx = torch.randperm(len(t))[:64]
    x = t[idx].unsqueeze(1)
    y = signal[idx].unsqueeze(1)
    info = trainer.train_step(x, y, nn.MSELoss())
    
    for i, block in enumerate(cms.blocks):
        total_norm = 0
        for p in block.parameters():
            if p.grad is not None:
                total_norm += p.grad.data.norm(2).item() ** 2
        grad_norms[f'Block {i} (C^{i}={4**i})'].append(total_norm ** 0.5)

fig, ax = plt.subplots(figsize=(10, 4))
for label, norms in grad_norms.items():
    ax.plot(norms, label=label, alpha=0.7)
ax.set_title('Gradient Norms per CMS Block')
ax.set_xlabel('Step')
ax.set_ylabel('Gradient L2 Norm')
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()
print('Fast blocks (low C) = noisy gradients, slow blocks (high C) = stable gradients')